In [ ]:
import os
import re
import glob
import matplotlib.pyplot as plt
import numpy as np
import cv2
import h5py
from pathlib import Path
from datetime import datetime

from PIL import Image as PILImage
import matplotlib.patches as patches

from MirrorExtractor.simple_mirror_extractor import SimpleMirrorExtractor
from MirrorFeatureExtractor.mirror_feature_extractor import MirrorFeatureExtractor

In [ ]:
def mark_mirrors_on_img(img_path, points_list):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img_rgb)

    for points in points_list:
        # === Polygon na podstawie 4 współrzędnych ===
        polygon = patches.Polygon(
                points,
                closed=True,
                linewidth=2,
                edgecolor='red',
                facecolor='red',
                alpha=0.3  # przezroczystość wypełnienia
            )
        ax.add_patch(polygon)

    plt.tight_layout()
    plt.show()

def get_feature_matrix_from_img(img_path):
    mirror_extractor = SimpleMirrorExtractor(str(MIRROR_POINTS_JSON))
    mirror_feature_extractor = MirrorFeatureExtractor()
    new_feature_matrix = np.zeros((249, 6))
    img_gray = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2GRAY)
    for mirror_id in range(0, 249):
        mirror_crop = mirror_extractor.extract_mirror_gray(img_gray, mirror_id=mirror_id)
        feat = mirror_feature_extractor.extract_texture_features(mirror_crop)
        new_feature_matrix[mirror_id, :] = list(feat.values())
    return new_feature_matrix

In [ ]:
MY_COMP_OUTPUT_DIR_PATH = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/data/images_for_analysis"
my_comp_root = Path("/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/data/2025/10/")

BASE_DIR = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data"
ROOT = Path(BASE_DIR) / "data/images_for_analysis"
MIRROR_POINTS_JSON = Path(BASE_DIR) / "points_IRCam.json"

In [ ]:
images_paths = sorted(glob.glob("/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/data/images_for_analysis/*.jpg"))
n_images = len(images_paths)
print(n_images)

mirror_extractor = SimpleMirrorExtractor(str(MIRROR_POINTS_JSON))
mirror_feature_extractor = MirrorFeatureExtractor()

feature_names = ['lbp_entropy', 'glcm_contrast', 'glcm_dissimilarity',
                 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation',
                 "sobel_mean", "soble_max",
                 "laplacian_mean", "laplacian_std", "edge_density"]

feature_matrix = np.zeros((n_images, 249, 11))  # wypełniasz w pętli po lustrach

for i, path in enumerate(images_paths[:]):
    img_gray = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2GRAY)
    for mirror_id in range(0, 249):
        mirror_crop = mirror_extractor.extract_mirror_gray(img_gray, mirror_id=mirror_id)
        feat_texture = mirror_feature_extractor.extract_texture_features(mirror_crop)
        feat_edge_gradient = mirror_feature_extractor.extract_edge_and_gradient_features(mirror_crop)
        feature_matrix[i, mirror_id, :6] = list(feat_texture.values())
        feature_matrix[i, mirror_id, 6:] = list(feat_edge_gradient.values())

In [ ]:
np.corrcoef(feature_matrix[:, 120, :])

In [ ]:
mirror_id_list = [m_id for m_id in range(0, 240, 40)]
plt.figure()
for m_id in mirror_id_list:
    plt.plot(feature_matrix[:120, m_id, 5], label=m_id)
    plt.legend()
plt.show()

In [ ]:
plt.plot(feature_matrix[:, 17, 9], feature_matrix[:, 17, 0], 'bo')

In [ ]:
IMG_PATH = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/data/images_for_analysis/IRCamM1T20251005_075000M.jpg"
IMG_PATH = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/data/2025/12/15/IRCamM1T20251215_081547M.jpg"
IMG_PATH = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/data/2025/12/15/IRCamM1T20251215_081547M.jpg"
IMG_PATH = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/data/2025/12/15/IRCamM1T20251215_084547M.jpg"
IMG_PATH = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/data/2025/12/17/IRCamM1T20251217_081547M.jpg"

img_rgb = cv2.imread(str(IMG_PATH), cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(cv2.imread(str(IMG_PATH)), cv2.COLOR_BGR2GRAY)

mirror_extractor = SimpleMirrorExtractor(str(MIRROR_POINTS_JSON))

p_list = []
for m_id in [  8,  10,  17,  18,  39,  40,  44,  45,  46,  47,  51,  57,  77,
         80,  91,  92,  97,  98, 109, 115, 116, 131, 132, 135, 148, 149,
        200, 212, 219, 220, 221, 222, 235, 236, 239, 244]:
    p_list.append(mirror_extractor.get_point_coords(m_id))
mark_mirrors_on_img(IMG_PATH, p_list)

In [ ]:
feature_matrix.shape

In [ ]:
! ls ../data/

In [ ]:
np.savez("../data/feature_matrix_1.npz",
         feature_matrix=feature_matrix,
         feature_names=feature_names)

In [ ]:
data = np.load("../data/feature_matrix_1.npz", allow_pickle=True)

In [ ]:
print(data["feature_matrix"].shape)
print(data["feature_names"].shape)

In [ ]:
f_matrix = feature_matrix[:120, :, :6]
baseline_median = np.median(f_matrix, axis=0)
baseline_mad = np.median(
    np.abs(f_matrix - baseline_median[np.newaxis, :, :]),
    axis=0
)*1.4826

#Minum MAD żeby uniknąć dzielenia przez 0
baseline_mad = np.maximum(baseline_mad, 1e-6) #

In [ ]:
new_feature_matrix = get_feature_matrix_from_img(IMG_PATH)
z_from_baseline = (new_feature_matrix - baseline_median) / baseline_mad

In [ ]:
z_from_baseline.shape

In [ ]:
np.where(z_from_baseline[:, 3] > 3.5)

In [ ]:
p_list = []
for m_id in [  0,   1,   3,   5,   6,   7,  10,  14,  17,  18,  23,  25,  29,
         30,  33,  34,  38,  39,  40,  41,  43,  45,  48,  51,  52,  53,
         55,  56,  58,  60,  63,  67,  68,  69,  73,  74,  76,  78,  80,
         81,  82,  89,  90,  92,  93,  96,  98, 100, 107, 109, 112, 113,
        114, 116, 117, 119, 123, 124, 129, 131, 132, 134, 135, 139, 148,
        149, 183, 184, 203, 206, 211, 215, 216, 219, 220, 221, 222, 225,
        231, 233, 234, 235, 236, 239, 244, 245]:
    p_list.append(mirror_extractor.get_point_coords(m_id))
mark_mirrors_on_img(IMG_PATH, p_list)